# 03 — Modeling, Evaluation & Hyperparameter Tuning

This is the Phase 3/4 core of the project: train multiple classical ML models on the
TEP fault diagnosis task, evaluate them properly, tune the best one, and save it.

Task: **multi-class classification** — predict `faultNumber` (0 = normal, 1–20 = fault
type) from the 52 process sensor readings.


In [1]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import xgboost as xgb

X_train = np.load("../data/processed/X_train.npy")
X_test  = np.load("../data/processed/X_test.npy")
y_train = np.load("../data/processed/y_train.npy")
y_test  = np.load("../data/processed/y_test.npy")
groups_train = np.load("../data/processed/groups_train.npy")

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Number of classes:", len(np.unique(y_train)))


X_train: (199857, 208)  X_test: (50143, 208)
Number of classes: 20


## Baseline: Logistic Regression

Always start with the simplest model. If a complex model can't beat this by a
meaningful margin, something is wrong with your features or pipeline.


In [3]:
baseline = LogisticRegression(max_iter=1000, n_jobs=-1)
baseline.fit(X_train, y_train)

pred_baseline = baseline.predict(X_test)
print("Baseline Logistic Regression")
print("Accuracy:", accuracy_score(y_test, pred_baseline))
print("Macro F1:", f1_score(y_test, pred_baseline, average='macro'))


Baseline Logistic Regression
Accuracy: 0.5403346429212452
Macro F1: 0.5212803627018221


## Random Forest

A strong, low-effort improvement over linear models — handles non-linear
relationships between sensor readings and fault types.


In [4]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
print("Random Forest")
print("Accuracy:", accuracy_score(y_test, pred_rf))
print("Macro F1:", f1_score(y_test, pred_rf, average='macro'))


Random Forest
Accuracy: 0.7016931575693517
Macro F1: 0.7241637904800117


## XGBoost

The model most likely to give you the best result here. Worth tuning properly.


In [6]:
# Shift labels from 1-20 down to 0-19
y_train = y_train - 1
y_test = y_test - 1

In [7]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)

pred_xgb = xgb_model.predict(X_test)
print("XGBoost (default-ish params)")
print("Accuracy:", accuracy_score(y_test, pred_xgb))
print("Macro F1:", f1_score(y_test, pred_xgb, average='macro'))


XGBoost (default-ish params)
Accuracy: 0.8083281814011926
Macro F1: 0.8171803221130363


## Proper Cross-Validation (Grouped by Simulation Run)

A single train/test split is noisy. Use GroupKFold so that, just like the original
split, no simulation run leaks across folds.


In [ ]:
gkf = GroupKFold(n_splits=5)

cv_scores = cross_val_score(
    xgb.XGBClassifier(n_estimators=200, max_depth=6, n_jobs=-1, random_state=42, eval_metric='mlogloss'),
    X_train, y_train,
    cv=gkf.split(X_train, y_train, groups=groups_train),
    scoring='f1_macro',
    n_jobs=1  # outer parallelism conflicts with inner n_jobs=-1, keep this at 1
)

print("Macro F1 across 5 grouped folds:", cv_scores)
print("Mean:", cv_scores.mean(), " Std:", cv_scores.std())


## Hyperparameter Tuning (XGBoost)

Using RandomizedSearchCV with a modest grid — runs in reasonable time on a laptop.
Increase `n_iter` if you have time/compute to spare.


In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}

search = RandomizedSearchCV(
    xgb.XGBClassifier(n_jobs=-1, random_state=42, eval_metric='mlogloss'),
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1_macro',
    cv=GroupKFold(n_splits=3).split(X_train, y_train, groups=groups_train),
    random_state=42,
    verbose=1,
    n_jobs=1
)

search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)

best_model = search.best_estimator_


## Final Evaluation on Held-Out Test Set


In [ ]:
final_pred = best_model.predict(X_test)

final_acc = accuracy_score(y_test, final_pred)
final_f1  = f1_score(y_test, final_pred, average='macro')

print(f"FINAL TEST ACCURACY: {final_acc:.4f}")
print(f"FINAL TEST MACRO F1: {final_f1:.4f}")
print()
print(classification_report(y_test, final_pred))


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_test, final_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues', square=True)
plt.title('Confusion Matrix — Fault Classification')
plt.xlabel('Predicted Fault Number')
plt.ylabel('True Fault Number')
plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# ── Feature importance — which sensors matter most for diagnosis ────────
feature_cols = joblib.load("../models/feature_cols.joblib")
importances = pd.Series(best_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(20).plot(kind='barh', color='#166534')
plt.title('Top 20 Most Important Features (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150)
plt.show()


In [ ]:
# ── Save the final model ─────────────────────────────────────────────────
joblib.dump(best_model, "../models/tep_fault_classifier_xgb.joblib")
print("Model saved to models/tep_fault_classifier_xgb.joblib")


## Resume Bullet — Fill In Your Real Numbers

Once you have your final accuracy / macro F1 from the cell above, here's a template:

> Built a multi-class fault diagnosis system on the Tennessee Eastman Process benchmark
> dataset (52 sensor variables, 21 operating conditions); engineered time-series features,
> trained and tuned XGBoost with grouped cross-validation to prevent data leakage across
> simulation runs, achieving **[YOUR macro F1]** macro F1 / **[YOUR accuracy]%** accuracy
> on held-out simulation runs.

This is a strong resume line because it shows: domain-specific application (ChemE +
ML), correct ML methodology (leakage-aware CV — most candidates get this wrong),
and a real, defensible metric. Be ready to explain GroupKFold and why a normal
train/test split would have leaked information — that single point will impress
interviewers more than the metric itself.

## What's Left Before Deep Learning (Phase 5)

You've now completed the hands-on core of Phases 1–4. To round things out before
moving to deep learning:
- Try the engineered features (`USE_ENGINEERED_FEATURES = True` in notebook 2) and
  compare the macro F1 — does feature engineering actually help here?
- Try PCA on the 52 features and visualize fault clusters in 2D — good for your
  portfolio and ties directly back to the linear algebra in Phase 2.
- Push this whole project to GitHub with a clear README explaining the leakage-safe
  split — that detail alone differentiates you from most other ML resumes.
